# base

> The reset, the document on the tokens, and prose

In [ ]:
#| default_exp base

The base layer styles native HTML elements on the tokens, so a page written as plain semantic markup is readable before any component is involved. It has three parts: a reset that removes the browser's default styling, document styles for text, links, code, lists, and tables, and typrose for long-form prose. Each registers with `reg.rule` in the `reset` or `base` layer.

In [ ]:
from fastcore.utils import *
from importlib.resources import files
from answerui.registry import reg
import answerui.tokens  # installs Registry.context

In [ ]:
from fasthtml.common import *
from answerui.demo import demo

Everything in this notebook has a light and a dark rendering, so a helper shows both at once. Each panel is an island with its own `color-scheme`, and the page and text colours re-derive inside it:

In [ ]:
def schemes(*fts):
    "The same markup in a light and a dark island, side by side"
    panels = [Div(*fts, style=f'color-scheme:{s};background:var(--n-3);color:var(--n2);padding:var(--space)') for s in ('light', 'dark')]
    return Div(*panels, style='display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))')

## Reset

The browser's default stylesheet puts margins on headings and paragraphs, gives form controls their own font, and lets images overflow. We undo these once, here. Margins go to zero on every block element, at zero specificity so any later rule wins. Spacing then comes only from a layout container or a prose region, so a parent controls the gaps between its children. Controls inherit the surrounding font and colour, and media is constrained to its container.

In [ ]:
reg.rule('reset', r"""
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
""", layer='reset', doc='Border-box sizing, no block margins, media constrained, controls inheriting text.')

reset [reset]: Border-box sizing, no block margins, media constrained, controls inheriting text.

## Document

The page takes the base rung and text the top of the neutral ladder, so the whole document follows `--hue` and `color-scheme`. Headings, links, and code read the type and colour tokens directly. `body` is a size container, so components can query the width they actually have rather than the viewport. `accent-color` and `caret-color` make native controls follow the hue before any control is styled.

Native scrollbars use the muted neutral for their thumb and a transparent track. They follow the surrounding palette without changing the browser's scrollbar width or behaviour. The document keeps a gutter for the vertical scrollbar whether or not the page scrolls, so moving between a short page and a long one does not shift the content sideways.

Scrolling the page, by a link to an anchor or a control taking focus, is smooth, and instant under a reduced-motion preference. Keyboard focus draws a ring, `--ring`, around the focused element; a component that draws focus around a larger box, such as a group around its input, reads the same value. A pressable control shows a pointer, and a disabled one the barred cursor.

A `mark` is a warning context, so its wash and ink come from the warning family.

In [ ]:
reg.context('mark', 'var(--hue-warning)')
reg.rule('document', r"""
:root { accent-color: var(--v); caret-color: var(--v); background: var(--n-3); --ring: 2px solid var(--v-1); }
html { scrollbar-gutter: stable; scroll-behavior: smooth; }
* { scrollbar-color: var(--n1) transparent; }
body { color: var(--n2); font: var(--weight) var(--text)/var(--leading) var(--font); container-type: inline-size; }
button, a[button], input:is([type=checkbox], [type=radio], [type=range]) { cursor: pointer; }
:disabled { cursor: not-allowed; }
h1, h2, h3, h4, h5, h6 { font-family: var(--font-heading, inherit); font-weight: var(--weight1); line-height: var(--leading-1); text-wrap: balance; }
h1 { font-size: var(--text3); } h2 { font-size: var(--text2); } h3 { font-size: var(--text1); }
p { text-wrap: pretty; }
a { color: var(--v); }
code, kbd, pre, samp { font-family: var(--font-mono); }
code, kbd { font-size: .875em; padding: .1em .3em; border-radius: var(--radius-1); }
code { background: var(--n-2); }
kbd { border: 1px solid var(--n-1); box-shadow: var(--raised); }
pre { background: var(--n-2); padding: var(--space); border-radius: var(--radius); overflow-x: auto; }
pre code { font-size: inherit; padding: 0; background: none; }
ul, ol { padding-inline-start: var(--space1); }
blockquote { border-inline-start: 3px solid var(--n-1); padding-inline-start: var(--space); color: var(--n1); }
hr { border: 0; border-block-start: 1px solid var(--n-1); }
small, figcaption { font-size: var(--text-1); }
figcaption { color: var(--n1); }
table {
  border-collapse: separate; border-spacing: 0;
  caption { caption-side: bottom; text-align: start; padding: var(--space-2) var(--space); font-size: var(--text-1); color: var(--n1); }
  :is(th, td) { padding: var(--space-2) var(--space); text-align: start; }
  thead th { font-weight: var(--weight1); border-block-end: 1px solid var(--n); }
  tbody tr + tr > * { border-block-start: 1px solid var(--n-1); }
  tfoot tr > * { border-block-start: 1px solid var(--n); font-weight: var(--weight1); }
  :is(th, td):is([num], [align=right]) { text-align: end; font-variant-numeric: tabular-nums; }
}
:focus-visible { outline: var(--ring); outline-offset: 2px; }
mark { background: var(--v-3); color: var(--v2); padding: .1em .2em; border-radius: var(--radius-1); }
@media (prefers-reduced-motion: reduce) { html { scroll-behavior: auto; } }
""", layer='base', doc='The page, text, headings, links, code, lists, quotes, tables, rules, marks, focus, and cursors on the tokens.')

document [base]: The page, text, headings, links, code, lists, quotes, tables, rules, marks, focus, and cursors on the tokens.

One more rule belongs to the page rather than to any component. A component sets its own `display`, and by layer order that beats the browser's rule for the `hidden` attribute, so a hidden card or alert would stay on screen. Native state has to win, so the attribute gets a rule in the utilities layer, above every component. The `until-found` form is left to the browser, which reveals such content when a find-in-page or fragment navigation lands in it:

In [ ]:
reg.rule('hidden', '[hidden]:not([hidden=until-found]) { display: none; }', layer='utilities', doc='The hidden attribute wins over any display a component sets.')

hidden [utilities]: The hidden attribute wins over any display a component sets.

A sample with nothing but content elements. None of them carries a margin, so the sample spaces its children with a grid gap, as a layout container would:

In [ ]:
sample = Div(
    H1('A document on the ladder'),
    P('Body text with ', A('a link', href='#'), ', ', Code('code'), ', ', Kbd('Ctrl'), ' and a ', Mark('marked'), ' phrase.'),
    H2('Lists, quotes, and tables'),
    Ul(Li('Bullets take the hairline rung'), Li('Captions and quotes take the muted rung')),
    Blockquote('Quoted text sits on a hairline and drops to the muted rung.'),
    Pre(Code('print("code blocks sit on the raised surface")')),
    Table(Thead(Tr(Th('Rung'), Th('Job'))), Tbody(Tr(Td('n2'), Td('text')), Tr(Td('v'), Td('links')))),
    Hr(), Small('Small print.'),
    style='display:grid;gap:var(--space)')
demo(schemes(sample))

```html
<div style="display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))">
  <div style="color-scheme:light;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div style="display:grid;gap:var(--space)">
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </div>
  </div>
  <div style="color-scheme:dark;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div style="display:grid;gap:var(--space)">
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </div>
  </div>
</div>

```

Headings use the same font as body text. To give the headings in part of a page a different font, set `--font-heading` there. The second article here uses `--font-serif`:


In [ ]:
demo(Article(H2('Account settings'), P('This heading is in the body face.')),
    Article(H2('A short history of type'), P('This one is in the serif, set by its article.'), style='--font-heading: var(--font-serif)'))


```html
<article>
  <h2>Account settings</h2>
  <p>This heading is in the body face.</p>
</article>
<article style="--font-heading: var(--font-serif)">
  <h2>A short history of type</h2>
  <p>This one is in the serif, set by its article.</p>
</article>

```

## Icons

Icons are small inline SVGs in text, links, and buttons. answerui takes them from [fastlucide](https://github.com/AnswerDotAI/fastlucide), the Lucide set for FastHTML. The base layer only makes an icon fit wherever it is placed.

A sprite icon is an `svg` that contains only a `use` element. The `use` points at a `symbol` in a hidden sheet elsewhere in the page, and the symbol holds the shape. fastlucide, like most icon sets, works this way, so each shape appears once however many times the page uses it. To place an icon, import its constructor from `fastlucide.icons` and call it. `reg.ftrs()` adds the sheet to the end of every page, and `demo` adds it to every preview. We recognise a sprite icon by its `use`, so it needs no attribute of ours. Mark an SVG drawn in full, such as a brand logo, with `icon` instead.

Both kinds inherit the surrounding text colour through `currentColor`. We also size them from the font size rather than from the `width` and `height` in their markup, so a heading, a button, and a caption each get an icon that fits their own line without being told a size. Setting `flex: none` stops a cluster or a button from squeezing the icon when space is short.

In [ ]:
reg.rule('icons', r'''
svg:is(:has(> use), [icon]) { inline-size: 1.25em; block-size: 1.25em; vertical-align: -.3em; flex: none; }
''', layer='base', doc='A sprite icon, or an inline SVG marked `icon`, sized and aligned to the text around it.')

icons [base]: A sprite icon, or an inline SVG marked `icon`, sized and aligned to the text around it.

Here fastlucide's `X` appears in a heading, a paragraph, and small print. The link holds a filled square marked `icon`:

In [ ]:
from fasthtml.svg import Path
from fastlucide.icons import X

In [ ]:
square = Svg(Path(d='M4 4h16v16H4z'), viewBox='0 0 24 24', fill='currentColor', icon=True)
demo(Div(H2(X(), ' A heading'), P(X(), ' Body text with ', A(square, ' a link', href='#'), '.'), Small(X(), ' Small print'), style='display:grid;gap:var(--space)'))

```html
<div style="display:grid;gap:var(--space)">
  <h2>
<svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg> A heading  </h2>
  <p>
<svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg> Body text with <a href="#"><svg viewbox="0 0 24 24" fill="currentColor" icon><path d="M4 4h16v16H4z"></path></svg> a link</a>.  </p>
<small><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg> Small print</small></div>

```

## Prose

Long-form content wants more than chrome: rhythm between blocks, a reading measure, and typographic care in lists, quotes, and code. That is [typrose](https://github.com/AnswerDotAI/typrose)'s job. It is a CSS-native port of Tailwind's typography plugin, scoped under a `prose` class, with every colour read from a `--prose-*` property on that scope.

We bundle the file and rewrite its selectors at registration, since the framework marks scopes with attributes: `.prose` becomes `[prose]` and a size variant such as `.prose-lg` becomes `[prose="lg"]`, quoted because a value like `2xl` is not valid unquoted, so `Article(prose=True)` and `Article(prose='lg')` both match. Its grey theme classes and the dark-mode `invert` block go at the same time, because the ladder does both of those jobs:

In [ ]:
TYPROSE = files('answerui').joinpath('css/typrose.css').read_text()
TYPROSE = re.sub(r'(/\*[^*]*\*/\n)?\.prose-(slate|zinc|neutral|stone|invert) \{[^}]*\}\n*', '', TYPROSE)
TYPROSE = re.sub(r'\.prose-(\w+)', r'[prose="\1"]', TYPROSE)
TYPROSE = re.sub(r'\.prose\b', '[prose]', TYPROSE)
reg.rule('typrose', TYPROSE, layer='base', doc='Prose typography from typrose, on the `prose` attribute.')

typrose [base]: Prose typography from typrose, on the `prose` attribute.

typrose's own colours are a fixed grey scale. Re-declaring its colour properties on `[prose]`, later in the same layer, maps each onto a rung, so prose follows the hue, the scheme, and any context it sits in. Two are left alone: the `hr` divider icon is an inline SVG with a fixed stroke, and the `kbd` shadow is a fixed black alpha. The same rule caps a prose region at a reading measure, which typrose leaves to the page, and sets its font size from `--text` times typrose's own size setting, in place of the fixed `1rem` typrose starts from. Its rhythm stays typrose's, tables included: `--prose-scale` spaces prose relative to its type, `--space` spaces chrome and layout, and `--text` is the base size the two share:

In [ ]:
reg.rule('prose', r"""
[prose] {
  max-inline-size: 65ch; font-size: calc(var(--text) * var(--prose-font-scale));
  --prose-body: var(--n2); --prose-headings: var(--n2); --prose-bold: var(--n2); --prose-quotes: var(--n2); --prose-code: var(--n2);
  --prose-links: var(--v);
  --prose-lead: var(--n1); --prose-counters: var(--n1); --prose-captions: var(--n1);
  --prose-bullets: var(--n-1); --prose-hr: var(--n-1); --prose-quote-borders: var(--n-1);
  --prose-th-borders: var(--n); --prose-td-borders: var(--n-1);
}
""", layer='base', doc='typrose colours mapped onto the ladder, its size from `--text`, and the reading measure.')

prose [base]: typrose colours mapped onto the ladder, its size from `--text`, and the reading measure.

The same content as an article, with rhythm and measure from typrose and colour from the ladder:

In [ ]:
article = Article(*sample.children, prose=True)
demo(schemes(article))

```html
<div style="display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))">
  <div style="color-scheme:light;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <article prose>
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </article>
  </div>
  <div style="color-scheme:dark;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <article prose>
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </article>
  </div>
</div>

```

Inside a `danger` context the whole article re-derives, because every colour it reads is a rung:

In [ ]:
demo(schemes(Div(article, danger=True)))

```html
<div style="display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))">
  <div style="color-scheme:light;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div danger>
      <article prose>
        <h1>A document on the ladder</h1>
        <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.        </p>
        <h2>Lists, quotes, and tables</h2>
        <ul>
          <li>Bullets take the hairline rung</li>
          <li>Captions and quotes take the muted rung</li>
        </ul>
        <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>        <table>
          <thead>
            <tr>
              <th>Rung</th>
              <th>Job</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td>n2</td>
              <td>text</td>
            </tr>
            <tr>
              <td>v</td>
              <td>links</td>
            </tr>
          </tbody>
        </table>
<hr><small>Small print.</small>      </article>
    </div>
  </div>
  <div style="color-scheme:dark;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div danger>
      <article prose>
        <h1>A document on the ladder</h1>
        <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.        </p>
        <h2>Lists, quotes, and tables</h2>
        <ul>
          <li>Bullets take the hairline rung</li>
          <li>Captions and quotes take the muted rung</li>
        </ul>
        <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>        <table>
          <thead>
            <tr>
              <th>Rung</th>
              <th>Job</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td>n2</td>
              <td>text</td>
            </tr>
            <tr>
              <td>v</td>
              <td>links</td>
            </tr>
          </tbody>
        </table>
<hr><small>Small print.</small>      </article>
    </div>
  </div>
</div>

```

The consumers agree. A compact region sets the two base tokens and reads `--text` for its own size, and its heading, control, and prose follow, with prose keeping its own rhythm:

In [ ]:
def region(title, style=''):
    "A heading, a paragraph, a native control, and a prose region"
    return Div(H2(title), P('Body text at the region\'s size.'), Input(placeholder='A native control'),
        Article(P('Prose sets its size from the same base and keeps its own rhythm.'), P('A second paragraph shows that rhythm.'), prose=True),
        style=f'font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);{style}')

demo(Div(region('Normal'), region('Compact', style='--space:.75rem;--text:.875rem'), style='display:grid;grid-template-columns:1fr 1fr'))

```html
<div style="display:grid;grid-template-columns:1fr 1fr">
  <div style="font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);">
    <h2>Normal</h2>
    <p>Body text at the region's size.</p>
    <input placeholder="A native control">
    <article prose>
      <p>Prose sets its size from the same base and keeps its own rhythm.</p>
      <p>A second paragraph shows that rhythm.</p>
    </article>
  </div>
  <div style="font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);--space:.75rem;--text:.875rem">
    <h2>Compact</h2>
    <p>Body text at the region's size.</p>
    <input placeholder="A native control">
    <article prose>
      <p>Prose sets its size from the same base and keeps its own rhythm.</p>
      <p>A second paragraph shows that rhythm.</p>
    </article>
  </div>
</div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()